In [1]:
import pandas as pd
import numpy as np
import duckdb
import random
from datetime import datetime, timedelta

# ==========================================
# STEP 1: GENERATE SYNTHETIC DATA
# ==========================================
np.random.seed(42)
n_customers = 500
n_policies = 1000
n_claims = 800

# 1. Dim_Customer
customers = pd.DataFrame({
    'CustomerID': range(1, n_customers + 1),
    'Age': np.random.randint(18, 80, n_customers),
    'Region': np.random.choice(['HK Island', 'Kowloon', 'New Territories'], n_customers),
    'BonusMalusTier': np.random.choice(['Low', 'Med', 'High'], n_customers, p=[0.6, 0.3, 0.1])
})

# 2. Dim_Policy
policies = pd.DataFrame({
    'PolicyID': range(1, n_policies + 1),
    'CustomerID': np.random.randint(1, n_customers + 1, n_policies),
    'StartDate': [datetime(2025, 1, 1) + timedelta(days=random.randint(0, 300)) for _ in range(n_policies)],
    'Exposure': np.random.uniform(0.1, 1.0, n_policies).round(2), # Partial year policies
    'BasePremium': np.random.uniform(2000, 8000, n_policies).round(2),
    'RatingRelativity': np.random.choice([0.8, 1.0, 1.2, 1.5, 2.0], n_policies)
})

# 3. Fact_Claims
claims = pd.DataFrame({
    'ClaimID': range(1, n_claims + 1),
    'PolicyID': np.random.randint(1, n_policies + 1, n_claims),
    'ClaimDate': [datetime(2025, 1, 1) + timedelta(days=random.randint(0, 330)) for _ in range(n_claims)],
    'PaidAmount': np.random.uniform(0, 50000, n_claims).round(2),
    'ReservedAmount': np.random.uniform(0, 20000, n_claims).round(2)
})

# ==========================================
# STEP 2: LOAD INTO DUCKDB
# ==========================================
# Create an in-memory DuckDB connection
con = duckdb.connect(database=':memory:')

# Register Pandas DataFrames as DuckDB tables
con.register('Dim_Customer', customers)
con.register('Dim_Policy', policies)
con.register('Fact_Claims', claims)

print("✅ Synthetic Data Warehouse successfully loaded into DuckDB!")
print(f"Tables available: {con.execute('SHOW TABLES').fetchall()}")

✅ Synthetic Data Warehouse successfully loaded into DuckDB!
Tables available: [('Dim_Customer',), ('Dim_Policy',), ('Fact_Claims',)]


In [2]:
# ==========================================
# STEP 3: ADVANCED SQL DATA MART CONSTRUCTION
# ==========================================

sql_query = """
WITH PolicyMetrics AS (
    -- 1. Core Joins & Aggregations: Calculate Earned Premium and Incurred Claims
    SELECT 
        p.PolicyID,
        c.Age,
        c.Region,
        c.BonusMalusTier,
        p.Exposure,
        p.BasePremium,
        p.RatingRelativity,
        -- Earned Premium = Base Premium * Exposure (simplified for PoC)
        (p.BasePremium * p.Exposure) AS EarnedPremium,
        -- Incurred Claims = Paid + Reserved (Handle policies with no claims via LEFT JOIN)
        COALESCE(SUM(f.PaidAmount + f.ReservedAmount), 0) AS IncurredClaims
    FROM Dim_Policy p
    LEFT JOIN Dim_Customer c ON p.CustomerID = c.CustomerID
    LEFT JOIN Fact_Claims f ON p.PolicyID = f.PolicyID
    GROUP BY 
        p.PolicyID, c.Age, c.Region, c.BonusMalusTier, 
        p.Exposure, p.BasePremium, p.RatingRelativity
),

LargestClaimPerPolicy AS (
    -- 2. Window Function: Identify the largest single claim per policy
    SELECT 
        PolicyID,
        PaidAmount AS LargestClaimAmount,
        ROW_NUMBER() OVER(PARTITION BY PolicyID ORDER BY PaidAmount DESC) as rn
    FROM Fact_Claims
)

SELECT 
    pm.PolicyID,
    pm.Age,
    pm.Region,
    pm.BonusMalusTier,
    pm.Exposure,
    pm.EarnedPremium,
    pm.IncurredClaims,
    
    -- 3. Actuarial Metric: Loss Ratio
    CASE 
        WHEN pm.EarnedPremium > 0 THEN (pm.IncurredClaims / pm.EarnedPremium) 
        ELSE 0 
    END AS LossRatio,

    -- 4. Window Function: Regional Portfolio Totals (Without collapsing rows)
    SUM(pm.EarnedPremium) OVER(PARTITION BY pm.Region) AS RegionTotalEarnedPremium,
    SUM(pm.IncurredClaims) OVER(PARTITION BY pm.Region) AS RegionTotalIncurredClaims,

    -- 5. Conditional Logic: Age Binning for Analytics
    CASE 
        WHEN pm.Age BETWEEN 18 AND 25 THEN '18-25'
        WHEN pm.Age BETWEEN 26 AND 35 THEN '26-35'
        WHEN pm.Age BETWEEN 36 AND 45 THEN '36-45'
        WHEN pm.Age BETWEEN 46 AND 55 THEN '46-55'
        WHEN pm.Age BETWEEN 56 AND 65 THEN '56-65'
        ELSE '66+'
    END AS AgeBand,

    -- Join the largest claim
    lc.LargestClaimAmount

FROM PolicyMetrics pm
LEFT JOIN LargestClaimPerPolicy lc 
    ON pm.PolicyID = lc.PolicyID AND lc.rn = 1; -- Ensure we only get the #1 largest claim
"""

# Execute the SQL query and fetch the result into a Pandas DataFrame
data_mart_df = con.execute(sql_query).fetchdf()

print("✅ Analytical Data Mart successfully built!")
print(f"Total rows in Data Mart: {len(data_mart_df)}")
print("\n📊 First 5 rows of the Data Mart:")
display(data_mart_df.head())

✅ Analytical Data Mart successfully built!
Total rows in Data Mart: 1000

📊 First 5 rows of the Data Mart:


,PolicyID,Age,Region,BonusMalusTier,Exposure,EarnedPremium,IncurredClaims,LossRatio,RegionTotalEarnedPremium,RegionTotalIncurredClaims,AgeBand,LargestClaimAmount
0,27,39,New Territories,Low,0.26,1108.6166,16345.71,14.744241,791852.2088,8321631.44,36-45,309.19
1,50,57,New Territories,Low,0.84,4202.0244,68358.28,16.267940,791852.2088,8321631.44,56-65,47856.74
2,52,77,New Territories,Low,0.74,3512.1880,57462.12,16.360776,791852.2088,8321631.44,66+,38793.86
3,143,50,New Territories,Med,0.84,3523.0272,51555.55,14.633878,791852.2088,8321631.44,46-55,41708.21
4,177,63,New Territories,Med,0.81,4276.3626,39246.96,9.177650,791852.2088,8321631.44,56-65,8084.73


In [3]:
# ==========================================
# STEP 4: EXPORT TO CSV FOR POWER BI
# ==========================================
output_file = 'day1_gi_portfolio_data_mart.csv'
data_mart_df.to_csv(output_file, index=False)

print(f"✅ Data Mart exported successfully to: {output_file}")
print("You can now import this CSV directly into Power BI for Day 3!")

✅ Data Mart exported successfully to: day1_gi_portfolio_data_mart.csv
You can now import this CSV directly into Power BI for Day 3!


In [4]:
# ==========================================
# STEP 1: RATING ADEQUACY & LOSS RATIO ANALYSIS
# ==========================================

# Define the target loss ratio for the portfolio (e.g., 65%)
target_loss_ratio = 0.65

sql_rating_adequacy = f"""
WITH PolicyMetrics AS (
    -- Calculate Earned Premium, Incurred Claims, and Expected Claims per policy
    SELECT 
        p.PolicyID,
        c.Age,
        c.Region,
        c.BonusMalusTier,
        p.Exposure,
        p.BasePremium,
        p.RatingRelativity,
        (p.BasePremium * p.Exposure) AS EarnedPremium,
        COALESCE(SUM(f.PaidAmount + f.ReservedAmount), 0) AS IncurredClaims,
        -- Expected Claims = Earned Premium * Relativity * Target Loss Ratio
        (p.BasePremium * p.Exposure * p.RatingRelativity * {target_loss_ratio}) AS ExpectedClaims
    FROM Dim_Policy p
    LEFT JOIN Dim_Customer c ON p.CustomerID = c.CustomerID
    LEFT JOIN Fact_Claims f ON p.PolicyID = f.PolicyID
    GROUP BY 
        p.PolicyID, c.Age, c.Region, c.BonusMalusTier, 
        p.Exposure, p.BasePremium, p.RatingRelativity
),

SegmentSummary AS (
    -- Aggregate by Region and Age Band for Portfolio Level Analysis
    SELECT 
        Region,
        CASE 
            WHEN Age BETWEEN 18 AND 25 THEN '18-25'
            WHEN Age BETWEEN 26 AND 35 THEN '26-35'
            WHEN Age BETWEEN 36 AND 45 THEN '36-45'
            WHEN Age BETWEEN 46 AND 55 THEN '46-55'
            WHEN Age BETWEEN 56 AND 65 THEN '56-65'
            ELSE '66+'
        END AS AgeBand,
        SUM(EarnedPremium) AS TotalEarnedPremium,
        SUM(IncurredClaims) AS TotalIncurredClaims,
        SUM(ExpectedClaims) AS TotalExpectedClaims
    FROM PolicyMetrics
    GROUP BY Region, AgeBand
)

SELECT 
    Region,
    AgeBand,
    ROUND(TotalEarnedPremium, 2) AS TotalEarnedPremium,
    ROUND(TotalIncurredClaims, 2) AS TotalIncurredClaims,
    ROUND(TotalExpectedClaims, 2) AS TotalExpectedClaims,
    
    -- 1. Loss Ratio Calculation
    ROUND(CASE WHEN TotalEarnedPremium > 0 THEN TotalIncurredClaims / TotalEarnedPremium ELSE 0 END, 4) AS LossRatio,
    
    -- 2. Rating Adequacy (Lift = Actual / Expected)
    ROUND(CASE WHEN TotalExpectedClaims > 0 THEN TotalIncurredClaims / TotalExpectedClaims ELSE 0 END, 4) AS Lift,
    
    -- 3. Profitability Status
    CASE 
        WHEN (CASE WHEN TotalEarnedPremium > 0 THEN TotalIncurredClaims / TotalEarnedPremium ELSE 0 END) > {target_loss_ratio} THEN 'Underperforming'
        ELSE 'Performing'
    END AS SegmentStatus

FROM SegmentSummary
ORDER BY Lift DESC;
"""

# Execute and display
rating_adequacy_df = con.execute(sql_rating_adequacy).fetchdf()

print("✅ Rating Adequacy Analysis Complete!")
print(f"📊 Total Segments Analyzed: {len(rating_adequacy_df)}")
print("\nTop 5 Underperforming Segments (Highest Lift):")
display(rating_adequacy_df.head())

✅ Rating Adequacy Analysis Complete!
📊 Total Segments Analyzed: 18

Top 5 Underperforming Segments (Highest Lift):


,Region,AgeBand,TotalEarnedPremium,TotalIncurredClaims,TotalExpectedClaims,LossRatio,Lift,SegmentStatus
0,HK Island,36-45,167977.54,2375842.42,140333.23,14.1438,16.9300,Underperforming
1,Kowloon,18-25,109430.38,1246540.59,88801.69,11.3912,14.0374,Underperforming
2,Kowloon,56-65,125010.04,1467434.95,106831.94,11.7385,13.7359,Underperforming
3,New Territories,26-35,66884.29,741684.20,54681.31,11.0891,13.5638,Underperforming
4,New Territories,36-45,150872.88,1781689.87,134385.65,11.8092,13.2580,Underperforming


In [5]:
# ==========================================
# STEP 2: MONTHLY TREND ANALYSIS (Window Functions)
# ==========================================

sql_trend_analysis = """
WITH MonthlyClaims AS (
    -- Aggregate claims by month
    SELECT 
        EXTRACT(MONTH FROM ClaimDate) AS ClaimMonth,
        SUM(PaidAmount + ReservedAmount) AS MonthlyIncurredClaims
    FROM Fact_Claims
    GROUP BY EXTRACT(MONTH FROM ClaimDate)
)

SELECT 
    ClaimMonth,
    ROUND(MonthlyIncurredClaims, 2) AS MonthlyIncurredClaims,
    
    -- 1. LAG Function: Get previous month's claims
    LAG(MonthlyIncurredClaims, 1) OVER(ORDER BY ClaimMonth) AS PrevMonthClaims,
    
    -- 2. Calculate Month-over-Month (MoM) Growth Rate
    ROUND(CASE 
        WHEN LAG(MonthlyIncurredClaims, 1) OVER(ORDER BY ClaimMonth) > 0 
        THEN (MonthlyIncurredClaims - LAG(MonthlyIncurredClaims, 1) OVER(ORDER BY ClaimMonth)) / LAG(MonthlyIncurredClaims, 1) OVER(ORDER BY ClaimMonth)
        ELSE 0 
    END, 4) AS MoM_Growth_Rate

FROM MonthlyClaims
ORDER BY ClaimMonth;
"""

# Execute and display
trend_df = con.execute(sql_trend_analysis).fetchdf()

print("✅ Monthly Trend Analysis Complete!")
print("\n📈 Month-over-Month Claim Trends:")
display(trend_df)

✅ Monthly Trend Analysis Complete!

📈 Month-over-Month Claim Trends:


,ClaimMonth,MonthlyIncurredClaims,PrevMonthClaims,MoM_Growth_Rate
0,1,1752667.20,NaN,0.0000
1,2,2557545.81,1752667.20,0.4592
2,3,2359067.86,2557545.81,-0.0776
3,4,2399823.34,2359067.86,0.0173
4,5,2143445.51,2399823.34,-0.1068
5,6,2774282.67,2143445.51,0.2943
6,7,3229048.99,2774282.67,0.1639
7,8,3405316.28,3229048.99,0.0546
8,9,2301455.20,3405316.28,-0.3242
9,10,2549661.12,2301455.20,0.1078


In [6]:
# ==========================================
# STEP 3: EXPORT TO CSV FOR POWER BI
# ==========================================
rating_adequacy_df.to_csv('day2_rating_adequacy.csv', index=False)
trend_df.to_csv('day2_monthly_trends.csv', index=False)

print("✅ Day 2 Analytics exported successfully!")
print("Files ready for Power BI:")
print("   - day2_rating_adequacy.csv")
print("   - day2_monthly_trends.csv")

✅ Day 2 Analytics exported successfully!
Files ready for Power BI:
   - day2_rating_adequacy.csv
   - day2_monthly_trends.csv
